# 🗂️ 20 · Sorting a cell's particles without labels: Jim-Net and DISCA
### Deep iterative clustering with model selection, label smoothing and a cluster-quality index — and discovering new classes

**BiologyToResearchV3** · Part E — Unsupervised & geometric deep learning (2020–2026) · Module 20 of 26

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearchV3/notebooks/20_unsupervised_clustering_disca.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU is enough (≈ 3 min); a T4 GPU is optional |
| **Core frameworks** | Python 3, PyTorch, scikit-learn (PCA, Gaussian mixtures), SciPy, NumPy, Matplotlib |
| **Prerequisites** | Modules 04 (GMM, EM, BIC, Hungarian accuracy, ARI), 05 (PyTorch), 13 (averaging), 15 (3D CNNs), 19 (deep alignment) |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | **Zeng *et al.* 2023** (PNAS — DISCA); **Zeng, Howe & Xu 2021** (ICCV — Jim-Net); **Nair *et al.* 2025** (Brief. Bioinform. — class discovery with semantic prototypes); Zhu *et al.* 2022 (ICIP — unsupervised multi-task learning) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> "DISCA-lite" is a small, faithful-in-spirit re-implementation of DISCA's loop on 16³ simulated subtomograms (2 nm
> voxels, 560 particles, 10 iterations). It does not reproduce the paper's network size, datasets or numbers.
> Densities come from real PDB entries via PolNet; offline, the notebook falls back to simple stand-in shapes.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** why clustering raw subtomograms fails (pose and noise dominate) and what a useful feature must ignore.
2. **Describe** Jim-Net's idea of training alignment and clustering *together* from GMM pseudo-labels.
3. **Implement** DISCA's generalized-EM loop: YOPO features with global max pooling after every layer → PCA →
   Gaussian mixtures for several K → **BIC** model selection → Hungarian relabelling → network training with **label
   smoothing**, rotation augmentation and a hinge loss.
4. **Compute** the distortion-based Davies–Bouldin index (DDBI) and use it to pick the best iteration without labels.
5. **Evaluate** clusterings with ARI and Hungarian accuracy, and turn clusters into structures.
6. **Run** a small *generalized class discovery* experiment: some classes labelled, others new (Nair *et al.* 2025).

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, a simulated mixture of five complexes | 10 min |
| 1 · Why raw clustering fails | PCA + GMM on voxels | 10 min |
| 2 · Jim-Net: aligning and clustering together | the algorithm and what it achieved | 15 min |
| 3 · DISCA-lite | YOPO, statistical fitting, BIC, label smoothing, DDBI — run 10 iterations | 45 min |
| 4 · From clusters to structures | composition, averages, naming the clusters | 15 min |
| 5 · Known and new classes | semi-supervised class discovery | 20 min |
| 📄 / ✅ / 🧠 / 🚀 | paper connections, checkpoint, discussion, going further | 15 min |

## 🧫 The biological question

The *Mycoplasma pneumoniae* dataset EMPIAR-10499 contains 65 tomograms of whole bacterial cells. Picking every
blob-like object yields about **200,000 subtomograms**: ribosomes, other complexes, membrane pieces, junk. Template
matching finds only what you already have a template for; supervised networks need thousands of hand labels per class;
and nobody knows in advance *how many* kinds of objects there are. DISCA (Zeng *et al.*, PNAS 2023) sorts such
collections **without templates or labels**, estimates the number of groups itself, and then lets each group be averaged
into a structure — "structural pattern mining" at the scale of whole cells.

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — fine, everything runs on a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed yet — the next cell installs it.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                         ("sklearn", "scikit-learn"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, confusion_matrix
from sklearn.mixture import GaussianMixture

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| PyTorch", torch.__version__, "| device", DEVICE, "| seed", SEED)

### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import struct
import urllib.request

POLNET_COMMIT = "73e613cc9162d26c7fd740ce3299bcb280a3e3b6"
POLNET_URL = ("https://raw.githubusercontent.com/anmartinezs/polnet/"
              + POLNET_COMMIT + "/data/templates/{kind}/{name}.{ext}")
CACHE_DIR = os.environ.get("CRYO_COURSE_CACHE", "course_data")
ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2277, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=5851, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3325, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=659, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=839, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1309, symmetry="C1"),
    # NB: PolNet's 4V94 map holds TWO TRiC complexes side by side (1.95 MDa ≈ 2 × ~1 MDa).
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1952, symmetry="C2"),
    "1bxn": dict(name="RuBisCO", organism="Cupriavidus necator", mass_kDa=280, symmetry="D4"),
    "1qvr": dict(name="ClpB chaperone", organism="Thermus thermophilus", mass_kDa=297, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=None, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=None, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=239, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=236, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=43, symmetry="C1"),
}
def _download(url: str, dest: str, timeout: float = 60.0) -> bool:
    """Download ``url`` to ``dest`` (no-op if it exists). Returns True on success."""
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return True
    os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r, open(dest + ".part", "wb") as f:
            f.write(r.read())
        os.replace(dest + ".part", dest)
        return True
    except Exception as exc:  # network blocked, offline, ...
        print(f"  ⚠️  download failed for {url.split('/')[-1]}: {type(exc).__name__}")
        return False


def read_mrc(path_or_bytes) -> tuple[np.ndarray, float]:
    """Minimal MRC2014 reader (modes 0, 1, 2, 6). Returns (array[z, y, x], voxel_size_Å).

    The MRC header is 1024 bytes of little-endian 32-bit words; words 1–3 are the
    grid size (nx, ny, nz), word 4 the data type, words 11–13 the cell size in Å,
    and word 24 the length of an optional extended header.
    """
    raw = path_or_bytes if isinstance(path_or_bytes, (bytes, bytearray)) else open(path_or_bytes, "rb").read()
    if raw[:2] == b"\x1f\x8b":  # gzip
        import gzip
        raw = gzip.decompress(raw)
    nx, ny, nz, mode = struct.unpack("<4i", raw[0:16])
    mx, my, mz = struct.unpack("<3i", raw[28:40])
    cella = struct.unpack("<3f", raw[40:52])
    mapc, mapr, maps = struct.unpack("<3i", raw[64:76])
    nsymbt = struct.unpack("<i", raw[92:96])[0]
    dtype = {0: np.int8, 1: np.int16, 2: np.float32, 6: np.uint16}[mode]
    data = np.frombuffer(raw, dtype=dtype, offset=1024 + nsymbt, count=nx * ny * nz)
    vol = data.reshape(nz, ny, nx).astype(np.float32)
    if (mapc, mapr, maps) != (1, 2, 3) and (mapc, mapr, maps) != (0, 0, 0):
        # reorder axes so that the array is [z, y, x]
        order = [3 - maps, 3 - mapr, 3 - mapc]
        vol = np.transpose(vol, np.argsort(order))
    voxel = float(cella[0] / mx) if mx > 0 else 1.0
    return vol, voxel


def fetch_zoo_map(pdb_id: str, voxel: str = "10A") -> tuple[np.ndarray, float, str]:
    """Return (density[z, y, x], voxel_size_Å, source) for a molecular-zoo entry.

    Tries the pinned PolNet template first; if the download fails (no internet),
    falls back to a *procedural* stand-in so the notebook still runs.
    """
    pdb_id = pdb_id.lower()
    dest = os.path.join(CACHE_DIR, f"polnet_{voxel}_{pdb_id}.mrc")
    url = POLNET_URL.format(kind=f"mrcs_{voxel}", name=pdb_id, ext="mrc")
    if _download(url, dest):
        vol, vs = read_mrc(dest)
        return vol, vs, f"PolNet template from PDB {pdb_id.upper()}"
    vol = procedural_molecule(pdb_id)
    return vol, 10.0, f"procedural stand-in for {pdb_id.upper()} (offline fallback)"


def procedural_molecule(pdb_id: str, size: int = 40) -> np.ndarray:
    """Offline fallback: a coarse shape with the right size/symmetry flavour (1 nm voxels)."""
    g = (np.arange(size) - size / 2 + 0.5)
    z, y, x = np.meshgrid(g, g, g, indexing="ij")
    r = np.sqrt(x**2 + y**2)

    def blob(cz, cy, cx, s):
        return np.exp(-((z - cz)**2 + (y - cy)**2 + (x - cx)**2) / (2 * s**2))
    pid = pdb_id.lower()
    if pid in ("4v4r", "4v7r", "5mrc"):          # two unequal subunits
        v = blob(-3, 0, 0, 7.5) + 0.8 * blob(6, 2, 1, 5.5)
    elif pid in ("3j9i", "6utj", "4cr2"):        # stacked rings / barrel
        v = np.exp(-((r - 5.5)**2) / 4.0) * (np.abs(z) < 7.5)
        if pid != "3j9i":
            v = v + 0.8 * blob(10, 0, 0, 4)
    elif pid == "4v94":                          # double ring with cavity
        v = np.exp(-((r - 7)**2) / 6.0) * (np.abs(z) < 8)
    elif pid in ("3cf3", "1qvr"):                # hexameric ring
        ang = np.arctan2(y, x)
        v = np.exp(-((r - 5)**2) / 3.0) * (np.abs(z) < 4) * (1 + 0.5 * np.cos(6 * ang))
    elif pid == "1bxn":                          # compact ball
        v = blob(0, 0, 0, 4.5)
    else:                                        # small asymmetric blob
        v = blob(0, 0, 0, 3) + 0.5 * blob(2, 2, 0, 2)
    return ndimage.gaussian_filter(v.astype(np.float32), 1.0)


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    PolNet templates are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    com = np.array(ndimage.center_of_mass(v))
    shift = np.array(v.shape) / 2 - 0.5 - com
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)
    dst0 = np.maximum((size - s) // 2, 0)
    n = np.minimum(s, size)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def missing_wedge_mask(shape, tilt_max_deg: float = 60.0) -> np.ndarray:
    """Binary Fourier mask of a single-axis tilt series about y (±tilt_max).

    A coefficient (k_z, k_y, k_x) is measured if |k_z| ≤ |k_x|·tan(tilt_max)
    — the formula M(ξ) = 1(|ξ_3| ≤ |ξ_1| tan θ) used by Xu et al. (2012).
    For 2D images [z, x] pass a 2-tuple shape.
    """
    k = freq_grid(shape)
    kz, kx = k[0], k[-1]
    m = np.abs(kz) <= np.abs(kx) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[(0,) * len(shape)] = True               # keep the mean (DC) term
    return m.astype(np.float32)


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def normalize(v: np.ndarray) -> np.ndarray:
    """Zero mean, unit standard deviation."""
    return ((v - v.mean()) / (v.std() + 1e-12)).astype(np.float32)


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    """Pearson correlation of two arrays (flattened)."""
    a = a.ravel() - a.mean()
    b = b.ravel() - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def hungarian_accuracy(y_true, y_pred) -> float:
    """Clustering accuracy after the best one-to-one relabelling (Hungarian algorithm)."""
    from scipy.optimize import linear_sum_assignment
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    ct = np.zeros((y_pred.max() + 1, y_true.max() + 1))
    np.add.at(ct, (y_pred, y_true), 1)
    r, c = linear_sum_assignment(-ct)
    return float(ct[r, c].sum() / len(y_true))


def show_orthoslices(vol, title: str = "", axes=None, cmap: str = "gray"):
    """Show the three central orthogonal slices (xy, xz, yz) of a volume."""
    import matplotlib.pyplot as plt
    if axes is None:
        _, axes = plt.subplots(1, 3, figsize=(7.5, 2.6))
    z, y, x = (np.array(vol.shape) // 2)
    for ax, im, lab in zip(axes, [vol[z], vol[:, y, :], vol[:, :, x]], ["xy (z mid)", "xz (y mid)", "yz (x mid)"]):
        ax.imshow(im, cmap=cmap)
        ax.set_title(f"{title} {lab}".strip(), fontsize=8)
        ax.axis("off")
    return axes
print("🧰 toolkit loaded")

### 0.1 A mixture of five complexes with unequal abundances

We simulate what a template-free picker would hand to DISCA: subtomograms of five complexes in random orientations,
slightly off-centre, with a ±60° missing wedge and noise at SNR 0.1. As in real cells, abundances are unequal — many
ribosomes, fewer of everything else. To keep training fast we bin the 1 nm maps 2× to **16³ voxels of 2 nm** (DISCA's
real-data subtomograms were 24³ voxels of ~1.3 nm after binning).

In [ ]:
# ── 0.1 Simulate the subtomogram collection ──
CLASSES = ["4v4r", "4cr2", "3j9i", "1bxn", "1qvr"]
COUNTS = [200, 120, 100, 80, 60]
NAMES = [ZOO[p]["name"] for p in CLASSES]
templates = {}
for pid in CLASSES:
    vol, vs, src = fetch_zoo_map(pid)
    templates[pid] = prepare_map(vol, size=32, sigma=1.0)
    print(f"{pid.upper()} {ZOO[pid]['name']:24s} ← {src}")


def bin2(v):
    """Average 2×2×2 blocks: 32³ at 1 nm → 16³ at 2 nm."""
    n = v.shape[0] // 2
    return v.reshape(n, 2, n, 2, n, 2).mean((1, 3, 5))


WEDGE16 = missing_wedge_mask((16, 16, 16), 60)
t0 = time.time()
clean, y_true, poses = [], [], []
for c, (pid, n) in enumerate(zip(CLASSES, COUNTS)):
    for _ in range(n):
        R = random_rotation(rng)
        clean.append(bin2(rotate_volume(templates[pid], R, rng.uniform(-2, 2, 3))))
        y_true.append(c); poses.append(R)
clean, y_true = np.stack(clean), np.array(y_true)
sigma_noise = np.sqrt(np.stack([apply_fourier_mask(v, WEDGE16) for v in clean]).var() / 0.1)
X = np.stack([apply_fourier_mask(apply_fourier_mask(v, WEDGE16) + rng.normal(0, sigma_noise, v.shape), WEDGE16)
              for v in clean]).astype(np.float32)
X = (X - X.mean()) / X.std()
print(f"{len(X)} subtomograms of {X.shape[1:]} voxels in {time.time() - t0:.1f} s; counts per class: {COUNTS}")
fig, axes = plt.subplots(2, 5, figsize=(9, 3.8))
for c in range(5):
    i = int(np.where(y_true == c)[0][0])
    axes[0, c].imshow(bin2(templates[CLASSES[c]]).sum(0)); axes[0, c].set_title(NAMES[c], fontsize=8)
    axes[1, c].imshow(X[i, 8]); axes[1, c].set_title("one noisy subtomogram", fontsize=7)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout(); plt.show()

---
## 1 · Why clustering raw subtomograms fails

📍 **Where we are.** Module 13 sorted particles by *aligning* them to references and clustering alignment scores, and
Module 14 discovered patterns by iterating alignment and clustering. Module 19 learned alignment with a network. Here we
learn **features** that make alignment unnecessary for sorting: features that are the same for every copy of a molecule,
whatever its pose.

First, the naive approach: treat each subtomogram as a vector of 4,096 voxel values, reduce it with PCA, and fit a
Gaussian mixture with the right number of classes.

In [ ]:
# ── 1.1 PCA + GMM on raw voxels ──
z_raw = PCA(16, random_state=SEED).fit_transform(X.reshape(len(X), -1))
lab_raw = GaussianMixture(5, covariance_type="full", random_state=SEED).fit_predict(z_raw)
print(f"raw voxels → PCA(16) → GMM(K=5): ARI {adjusted_rand_score(y_true, lab_raw):.2f}, "
      f"Hungarian accuracy {hungarian_accuracy(y_true, lab_raw):.2f}")
fig, ax = plt.subplots(figsize=(5.2, 4))
for c in range(5):
    m = y_true == c
    ax.scatter(z_raw[m, 0], z_raw[m, 1], s=6, label=NAMES[c])
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2"); ax.legend(fontsize=7, markerscale=2)
ax.set_title("Raw voxels: classes overlap (pose and noise dominate)"); plt.show()

🔍 Even told the correct number of classes, clustering raw voxels mixes the molecules: the biggest differences between
two subtomograms are *where* and *how* the molecule sits and the noise, not *which* molecule it is. We need features
that are **invariant** to translation and rotation and robust to noise.

---
## 2 · Jim-Net: aligning and clustering at the same time

**Jim-Net** (Zeng, Howe & Xu, ICCV 2021) attacked the problem by training one network with two branches on shared
features: an **alignment branch** (coarse-to-fine, like Gum-Net in Module 19) and a **clustering branch**. Labels come from
the data itself (Algorithm 1 of the paper):

```text
repeat m times:
    f ← features of every image paired with itself       (shared feature extractor)
    fit a Gaussian mixture G to f; ℓ ← G(f)               (pseudo-labels)
    P ← all pairs (s, t) with ℓ_s = ℓ_t                   (same-cluster pairs)
    for (s, t, ℓ_s) in P: update the network with
        L = CE(clustering branch(F(s,t)), ℓ_s) + Σ_i γ_i · L_align(stage i)   (alignment loss: 1 − constrained correlation)
```

*In words:* clustering tells the alignment branch which pairs are worth aligning; alignment makes the shared features
pose-aware, which makes clustering better. On simulated subtomograms (the Gum-Net benchmark), joint training raised
clustering accuracy from 77.5 % (clustering branch alone) to 99.7 % at SNR 0.1, and from 51.3 % to 85.5 % at SNR 0.03.
On a 7-class air–water-interface dataset Jim-Net reached 96.6 % accuracy (PICA 60.5 %, DeepCluster 57.7 %). We do not
re-implement Jim-Net here (it needs both a Gum-Net-like aligner and a classifier); DISCA, next, reaches invariance in
a different and simpler way.

---
## 3 · DISCA-lite: deep iterative subtomogram clustering

DISCA alternates two steps, like the EM algorithm of Module 04 — a **generalized EM** loop:

```text
 ┌────────────► 1. features  = YOPO(subtomograms)                    (CNN, invariant to shifts)
 │              2. statistical fitting: PCA(16) → GMM for every candidate K → keep lowest BIC
 │              3. relabel clusters consistently (Hungarian); stop if labels stop changing
 │              4. score the clustering with DDBI; remember the best iteration
 └─────────────── 5. train YOPO to predict the cluster labels (smoothed labels, rotated copies, hinge loss)
```

### 3.1 YOPO — "You Only Pool Once"

Ordinary CNNs pool between convolutions and lose fine detail early. DISCA's **YOPO** feature extractor uses *no pooling
between convolutions*. Instead, after **every** convolution layer it takes a **global max pool** — the largest response
of each filter anywhere in the volume — and concatenates these numbers from all layers into one feature vector, followed
by batch normalisation and a dense layer. A global maximum does not care *where* a pattern occurs, so the features are
**translation-invariant** by construction; rotation invariance is encouraged by training on randomly rotated copies.
The input is multiplied by Gaussian noise (*Gaussian dropout*) during training, a regulariser for very noisy data.

*(The paper's YOPO: 10 Conv3D 3³ layers with 64–208 filters, ELU + batch norm, on 24³ inputs, then a 1024-unit dense
layer. Ours: 4 layers with 16–40 filters on 16³, then 32 units.)*

In [ ]:
# ── 3.1 YOPO feature extractor ──
class YOPO(nn.Module):
    """Valid 3D convolutions (no pooling in between); global max pooling after EVERY layer; concatenate → BN → dense."""

    def __init__(self, channels=(16, 24, 32, 40), n_features=32):
        super().__init__()
        layers, c_in = [], 1
        for c in channels:
            layers.append(nn.Sequential(nn.Conv3d(c_in, c, 3), nn.ELU(), nn.BatchNorm3d(c)))
            c_in = c
        self.layers = nn.ModuleList(layers)
        self.bn = nn.BatchNorm1d(sum(channels))
        self.fc = nn.Linear(sum(channels), n_features)

    def forward(self, x):
        pooled = []
        for layer in self.layers:
            x = layer(x)
            pooled.append(x.amax(dim=(2, 3, 4)))          # global max pool: one number per filter
        return self.fc(self.bn(torch.cat(pooled, 1)))


def extract_features(net, X, batch=128):
    """Feature vectors (n × 32) for all subtomograms, network in evaluation mode."""
    net.eval()
    with torch.no_grad():
        return torch.cat([net(torch.tensor(X[i:i + batch])[:, None]) for i in range(0, len(X), batch)]).numpy()


torch.manual_seed(SEED)
yopo = YOPO()
x_test = torch.tensor(X[:1])[:, None]
with torch.no_grad():
    yopo.eval()
    f_a = yopo(x_test)
    f_b = yopo(torch.roll(x_test, shifts=(2, -1, 1), dims=(2, 3, 4)))   # the same subtomogram, shifted
print(f"YOPO: {sum(p.numel() for p in yopo.parameters()):,} parameters; "
      f"feature change after a shift: {float((f_a - f_b).norm() / f_a.norm()):.3f} (relative)")

🔍 Shifting the input barely changes the features (only voxels entering or leaving the box matter) — translation
invariance from global max pooling, before any training.

### 3.2 Statistical fitting and model selection

At each iteration the features are reduced to 16 dimensions with PCA and modelled with a full-covariance Gaussian
mixture,

$$f(x_n)=\sum_{k=1}^{K}\phi_k\,\mathcal N(x_n;\mu_k,\Sigma_k),\qquad \hat k_n=\arg\max_k \frac{\phi_k\mathcal N(x_n;\mu_k,\Sigma_k)}{\sum_i\phi_i\mathcal N(x_n;\mu_i,\Sigma_i)} ,$$

for each candidate number of clusters $K$; the $K$ with the lowest **Bayesian information criterion** (BIC, Module 04)
wins. For the $K$ used in the previous iteration DISCA fits *two* mixtures — one warm-started from the previous labels,
one from scratch — and keeps the better BIC (the paper compares this to ε-greedy exploration). Empty clusters are
dropped, and when $K$ is unchanged the new labels are matched to the old ones with the Hungarian algorithm so that
"cluster 3" keeps meaning the same thing (this relabelling step is in the released code).

**Candidate K.** The paper searched $K$ from 2 to 20; the released code's default is {5, 10, 20}, i.e. it favours
*over-partitioning* (splitting a class into several clusters is harmless — they can be merged after averaging — whereas
merging two classes is not). We use {4, 5, 6, 8, 10}.

In [ ]:
# ── 3.2 Statistical fitting: PCA → GMMs → BIC; Hungarian relabelling; DDBI ──
CANDIDATE_KS = [4, 5, 6, 8, 10]
REG = 1e-4


def statistical_fitting(features, prev_labels, prev_K, seed):
    """Return (labels, K, pca_features, BIC table). Mirrors DISCA's util.statistical_fitting, simplified."""
    z = PCA(16, random_state=seed).fit_transform(features)
    fits = []
    for k in CANDIDATE_KS:
        g = GaussianMixture(k, covariance_type="full", reg_covar=REG, n_init=2, random_state=seed).fit(z)
        fits.append((g.bic(z), k, g.predict(z), "fresh"))
    if prev_K is not None:                                      # warm start from the previous labels
        means = np.stack([z[prev_labels == j].mean(0) for j in range(prev_K)])
        g = GaussianMixture(prev_K, covariance_type="full", reg_covar=REG, means_init=means,
                            random_state=seed).fit(z)
        fits.append((g.bic(z), prev_K, g.predict(z), "warm"))
    bic, K, labels, how = min(fits, key=lambda f: f[0])
    used = np.unique(labels)                                    # drop empty clusters
    labels = np.searchsorted(used, labels); K = len(used)
    if prev_K == K:                                             # keep cluster names stable (Hungarian matching)
        ct = np.zeros((K, K)); np.add.at(ct, (labels, prev_labels), 1)
        r, c = linear_sum_assignment(-ct); mapping = dict(zip(r, c))
        labels = np.array([mapping[l] for l in labels])
    return labels, K, z, {k: b for b, k, _, h in fits if h == "fresh"}


def ddbi(z, labels):
    """Distortion-based Davies–Bouldin index (as in DISCA.py): lower = tighter, better-separated clusters."""
    ks = np.unique(labels)
    mu = np.array([z[labels == k].mean(0) for k in ks])
    prec = np.array([np.linalg.inv(np.cov(z[labels == k].T) + REG * np.eye(z.shape[1])) for k in ks])
    T = np.array([np.mean(np.einsum("ni,ij,nj->n", z[labels == k] - mu[i], prec[i], z[labels == k] - mu[i]))
                  for i, k in enumerate(ks)])
    D = np.array([[(mu[j] - mu[i]) @ prec[i] @ (mu[j] - mu[i]) for j in range(len(ks))] for i in range(len(ks))])
    R = np.zeros((len(ks), len(ks)))
    for i in range(len(ks)):
        for j in range(len(ks)):
            if i != j:
                R[i, j] = (T[i] + T[j]) / (D[i, j] + D[j, i])
    return float(np.mean(R.max(0)))


print("statistical fitting and DDBI ready")

The **distortion-based Davies–Bouldin index** compares how spread out each cluster is, $T_k$ (the average Mahalanobis
distance of its members to its centre), with how far apart cluster centres are, $D_{kl}$:

$$T_k=\frac1{|C_k|}\sum_{n\in C_k}(x_n-\mu_k)^{\mathsf T}\Sigma_k^{-1}(x_n-\mu_k),\quad D_{kl}=(\mu_l-\mu_k)^{\mathsf T}\Sigma_k^{-1}(\mu_l-\mu_k),\quad R_{kl}=\frac{T_k+T_l}{D_{kl}+D_{lk}} ,$$

and DDBI averages the worst ratio over clusters. *In words:* small when clusters are compact and far apart. Because it
needs no ground truth, DISCA uses it to decide which iteration's labels and model to keep.

### 3.3 Training the network on its own pseudo-labels

The M-step trains YOPO plus a softmax output layer to predict the current cluster labels, with three safeguards against
trusting noisy labels too much:

- **Label smoothing:** $\ell_{ls}=(1-\alpha)\,\ell_{\text{one-hot}}+\alpha/K$, with $\alpha=0.2$ at the start, multiplied
  by 0.9 every iteration (labels get sharper as they become more reliable);
- **Rotation augmentation:** every subtomogram also appears as a randomly rotated copy with the same label, empty corners
  filled with noise. *(We use proper rotations, det = +1; the released code's SVD-based helper can produce mirror images —
  see Module 02.)*
- **Gaussian dropout** on the input and a **categorical hinge loss**,
  $\mathcal L=\max\big(0,\ 1+\max_{j\neq y}p_j-p_y\big)$ (generalised to smoothed labels).

When $K$ changes, a fresh output layer is attached.

In [ ]:
# ── 3.3 Augmentation and loss ──
def rotate_with_noise_fill(x, rng):
    """Proper random rotation of a normalised subtomogram; corners that rotate in from outside get N(0,1) noise."""
    R = random_rotation(rng)
    v = rotate_volume(x, R)
    inside = rotate_volume(np.ones_like(x), R) > 0.5
    return np.where(inside, v, rng.normal(0, 1, x.shape)).astype(np.float32)


def categorical_hinge(probs, targets):
    """Keras' categorical hinge: max(0, max_{non-target} p − p_target + 1), with (smoothed) target vectors."""
    pos = (targets * probs).sum(1)
    neg = ((1 - targets) * probs).max(1).values
    return F.relu(neg - pos + 1).mean()


def train_one_epoch(net, head, X, labels, K, alpha, rng, lr=1e-3, batch=32):
    """M-step: one epoch over the subtomograms and their rotated copies, with smoothed labels."""
    targets = (1 - alpha) * np.eye(K)[labels] + alpha / K
    Xa = np.concatenate([X, np.stack([rotate_with_noise_fill(x, rng) for x in X])])
    Ta = np.concatenate([targets, targets]).astype(np.float32)
    opt = torch.optim.Adam(list(net.parameters()) + list(head.parameters()), lr=lr)
    net.train(); perm = rng.permutation(len(Xa))
    for b in range(0, len(Xa), batch):
        idx = perm[b:b + batch]
        xb = torch.tensor(Xa[idx])[:, None]
        xb = xb * (1 + 0.5 * torch.randn_like(xb))              # Gaussian dropout (multiplicative noise)
        loss = categorical_hinge(F.softmax(head(net(xb)), 1), torch.tensor(Ta[idx]))
        opt.zero_grad(); loss.backward(); opt.step()
    return loss.item()


print("M-step ready")

### 3.4 Run the loop

Ten iterations; at each one we record the chosen $K$, the DDBI, and — because this is a simulation — the ARI and
Hungarian accuracy against the hidden truth (never used by the algorithm). We stop early if more than 99.9 % of the
labels are unchanged, as DISCA does.

In [ ]:
# ── 3.4 The DISCA-lite loop ──
torch.manual_seed(SEED)
yopo = YOPO()
head, labels, K = None, None, None
alpha = 0.2
log, best = [], dict(ddbi=np.inf)
t_start = time.time()
for it in range(10):
    t0 = time.time()
    feats = extract_features(yopo, X)
    new_labels, new_K, z, bic_table = statistical_fitting(feats, labels, K, seed=it)
    changed = 1.0 if labels is None or new_K != K else float(np.mean(new_labels != labels))
    labels, K_prev, K = new_labels, K, new_K
    score = ddbi(z, labels)
    log.append(dict(it=it, K=K, ddbi=score, ari=adjusted_rand_score(y_true, labels),
                    acc=hungarian_accuracy(y_true, labels), changed=changed, bic=bic_table))
    if score < best["ddbi"]:
        best = dict(ddbi=score, it=it, labels=labels.copy(), z=z.copy(),
                    state={k: v.clone() for k, v in yopo.state_dict().items()})
    if head is None or K != K_prev:
        head = nn.Linear(32, K)                                  # new output layer when K changes
    if changed < 0.001:
        print(f"iteration {it}: labels stable (>99.9 % unchanged) — stopping"); break
    train_one_epoch(yopo, head, X, labels, K, alpha, rng)
    alpha *= 0.9
    r = log[-1]
    print(f"iteration {it}: K = {K}, DDBI = {r['ddbi']:.2f}, ARI = {r['ari']:.2f}, accuracy = {r['acc']:.2f}, "
          f"labels changed {100 * changed:.0f} %  ({time.time() - t0:.1f} s)")
print(f"total {time.time() - t_start:.0f} s; lowest DDBI at iteration {best['it']}")

In [ ]:
# ── 3.5 How the loop behaved ──
its = [r["it"] for r in log]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.1))
axes[0].plot(its, [r["ari"] for r in log], "o-", label="ARI")
axes[0].plot(its, [r["acc"] for r in log], "s-", label="Hungarian accuracy")
axes[0].axhline(adjusted_rand_score(y_true, lab_raw), color="gray", ls="--", lw=1, label="raw-voxel GMM (ARI)")
axes[0].set_xlabel("iteration"); axes[0].set_ylim(0, 1.02); axes[0].legend(fontsize=7); axes[0].set_title("agreement with the truth")
axes[1].plot(its, [r["ddbi"] for r in log], "o-", color="tab:red"); axes[1].axvline(best["it"], color="gray", ls=":")
axes[1].set_xlabel("iteration"); axes[1].set_ylabel("DDBI (lower = better)"); axes[1].set_title("label-free quality score")
last_bic = log[-1]["bic"]
axes[2].plot(list(last_bic), list(last_bic.values()), "o-", color="tab:purple")
axes[2].set_xlabel("number of clusters K"); axes[2].set_ylabel("BIC"); axes[2].set_title(f"BIC at iteration {log[-1]['it']}")
plt.tight_layout(); plt.show()

🔍 **What just happened?** At iteration 0 the features come from an *untrained* network and the clusters are poor —
worse than raw voxels. Each M-step teaches YOPO to separate the current clusters; the next E-step then finds cleaner
clusters; and so on. Agreement with the hidden truth rises over the iterations — well above the raw-voxel baseline —
while the label-free DDBI trends downward (with ups and downs), so the algorithm can tell, without labels, which
iteration to trust: here the lowest DDBI coincides with one of the best ARIs. BIC settles on 4 clusters for our 5 classes,
and Section 4 shows which ones merge: similar-looking classes — the 20S proteasome with the 26S proteasome (whose core
*is* a 20S-type barrel), and RuBisCO with ClpB (both ~300 kDa blobs at 2 nm). This is exactly why the released code
prefers to over-partition and merge later.

✏️ **Your turn.** (1) Change `CANDIDATE_KS` to `[2, 3, 4, 5]` and re-run 3.4. What happens when an untrained network's
features suggest too few clusters? (2) Remove label smoothing (`alpha = 0.0`) or the rotated copies and compare the ARI
curve.

---
## 4 · From clusters to structures

A cluster is only a hypothesis; it becomes biology when its members are **aligned and averaged** into a structure that
can be recognised (DISCA used RELION for this step; in this course see Modules 11–13). Here we inspect the best
iteration's clusters three ways: what they contain (only possible in a simulation), what a naive unaligned average looks
like, and an *oracle* average that uses the true poses to show what a good alignment would reveal.

In [ ]:
# ── 4.1 Composition of each cluster (best-DDBI iteration) ──
lab_best = best["labels"]
K_best = lab_best.max() + 1
cm = confusion_matrix(y_true, lab_best, labels=range(max(5, K_best)))[:5, :K_best]
fig, ax = plt.subplots(figsize=(6.5, 3.2))
bottom = np.zeros(K_best)
for c in range(5):
    ax.bar(range(K_best), cm[c], bottom=bottom, label=NAMES[c]); bottom += cm[c]
ax.set_xlabel("cluster"); ax.set_ylabel("subtomograms"); ax.legend(fontsize=7)
ax.set_title(f"What each DISCA-lite cluster contains (iteration {best['it']}, ARI {adjusted_rand_score(y_true, lab_best):.2f})")
plt.show()

In [ ]:
# ── 4.2 Naive and oracle-aligned averages; naming clusters by correlation with the known structures ──
fig, axes = plt.subplots(2, K_best, figsize=(1.9 * K_best, 3.8))
for k in range(K_best):
    members = np.where(lab_best == k)[0]
    naive = X[members].mean(0)
    oracle = np.mean([rotate_volume(X[i], poses[i].T) for i in members], axis=0)   # undo the TRUE orientation
    scores = [pearson(oracle, bin2(templates[p])) for p in CLASSES]
    axes[0, k].imshow(naive.sum(0)); axes[0, k].set_title(f"cluster {k}: n = {len(members)}\nunaligned average", fontsize=7)
    axes[1, k].imshow(oracle.sum(0))
    axes[1, k].set_title(f"aligned (oracle)\nbest match: {NAMES[int(np.argmax(scores))]}\nr = {max(scores):.2f}", fontsize=7)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout(); plt.show()

🔍 Unaligned averages are featureless blobs — sorting is only half the job. Once members are brought into register
(here with the true poses, as a stand-in for real alignment), each cluster's average resembles one known complex; a
mixed cluster gives a blurred, ambiguous average. In the paper, clusters from *M. pneumoniae* yielded a 14.17 Å ribosome
average, and across five datasets DISCA recovered ribosomes, TRiC, capped proteasomes and phycobilisome arrays
(1.2–4.5 MDa) plus cellular features such as thylakoid and mitochondrial membranes and calcium-phosphate precipitates.

---
## 5 · Known and new classes: generalized class discovery

In practice we often *do* know some molecules (ribosomes, proteasomes) and could label a few, but every new cell type also
contains complexes nobody has labelled. **Generalized class discovery** asks for one clustering of *all* particles in which
known ones go to their classes and unknown ones form **new** clusters. **Nair *et al.* (2025)** tackled this for cryo-ET
with "auxiliary semantic prototypes": a 3D vision transformer inflated from a 2D self-supervised model, class-name text
embeddings from a language-vision model refined by a graph network, and prototype-based semi-supervised clustering.

Our miniature keeps the clustering stage: **semi-supervised k-means** on DISCA-lite's features — labelled particles of the
known classes are pinned to their class; centroids for the known classes start at the labelled means; extra centroids
for new classes are found from the data. We compare it with plain k-means and report accuracy on All, Known and New
particles (Hungarian-matched, as in the paper).

In [ ]:
# ── 5.1 Semi-supervised k-means on the learned features ──
KNOWN = [0, 1, 2]                    # ribosome, 26S proteasome, 20S proteasome have labels; RuBisCO and ClpB are "new"
z_best = best["z"]
labelled = np.zeros(len(y_true), bool)
for c in KNOWN:
    idx = np.where(y_true == c)[0]
    labelled[rng.choice(idx, size=len(idx) // 2, replace=False)] = True   # half of each known class is labelled


def semi_supervised_kmeans(z, labelled, y_lab, n_known, n_new, n_iter=50, seed=0):
    """k-means where labelled points keep their class; centroids for known classes start at the labelled means,
    centroids for new classes are seeded k-means++-style among the unlabelled points."""
    r = np.random.default_rng(seed)
    cents = [z[labelled & (y_lab == c)].mean(0) for c in range(n_known)]
    pool = z[~labelled]
    for _ in range(n_new):
        d2 = np.min([((pool - c) ** 2).sum(1) for c in cents], axis=0)
        cents.append(pool[r.choice(len(pool), p=d2 / d2.sum())])
    cents = np.array(cents)
    assign = np.zeros(len(z), int)
    for _ in range(n_iter):
        d = np.linalg.norm(z[:, None] - cents[None], axis=-1)
        assign = d.argmin(1)
        assign[labelled] = y_lab[labelled]                       # labelled points are pinned
        cents = np.array([z[assign == k].mean(0) if np.any(assign == k) else cents[k] for k in range(len(cents))])
    return assign


def all_known_new(y, pred, known):
    """Hungarian accuracy on all particles, on those of known classes, and on those of new classes (one matching)."""
    ct = np.zeros((pred.max() + 1, y.max() + 1)); np.add.at(ct, (pred, y), 1)
    r, c = linear_sum_assignment(-ct); mapping = dict(zip(r, c))
    mapped = np.array([mapping.get(p, -1) for p in pred])
    k = np.isin(y, known)
    return [float(np.mean(mapped[m] == y[m])) for m in (np.ones_like(k), k, ~k)]


pred_ss = semi_supervised_kmeans(z_best, labelled, y_true, n_known=3, n_new=2)
pred_km = KMeans(5, n_init=10, random_state=SEED).fit_predict(z_best)
unl = ~labelled
for name, pred in [("plain k-means (K = 5)", pred_km), ("semi-supervised k-means", pred_ss)]:
    a, kn, nw = all_known_new(y_true[unl], pred[unl], KNOWN)
    print(f"{name:25s}  All {100 * a:5.1f} %   Known {100 * kn:5.1f} %   New {100 * nw:5.1f} %   (unlabelled particles)")

🔍 Pinning labelled particles anchors the known classes, which helps the **Known** accuracy; the **New** classes depend
entirely on how well the features separate them — in our toy the gain over plain k-means is modest because DISCA-lite's
features already separate most classes. On real data Nair *et al.* report, for their first
known/new split, All/Known/New accuracies of 74.43/99.12/49.75 % on the 7-class "Noble" dataset and
89.99/99.94/80.04 % on the 5-class "Qiang" dataset, compared with 57.55/83.33/31.78 % and 76.01/86.82/65.20 % for the
strongest baseline (PromptCAL). As in our toy, discovering the *new* classes remains the hard part.

✏️ **Your turn.** Make RuBisCO a known class and the 26S proteasome new (`KNOWN = [0, 2, 3]`). Are some classes easier
to discover than others? Why?

---
### 📄 Paper connection — DISCA (Zeng *et al.*, PNAS 2023)

**Zeng X, Kahng A, Xue L, Mahamid J, Chang Y-W, Xu M.** *High-throughput cryo-ET structural pattern mining by unsupervised
deep iterative subtomogram clustering.* PNAS 120(15):e2213149120 (2023).
[doi:10.1073/pnas.2213149120](https://doi.org/10.1073/pnas.2213149120). Preprint title: *DISCA: high-throughput cryo-ET
structural pattern mining by deep unsupervised clustering*, bioRxiv 10.1101/2021.05.16.444381.
- **Question:** sort hundreds of thousands of template-free picked subtomograms into structurally homogeneous groups —
  and estimate how many groups exist — without templates or labels.
- **What the paper does:** the generalized-EM loop above: YOPO features (10 valid 3D conv layers, global max pooling after
  each, concatenation, dense 1024); PCA(16) + full-covariance GMMs over candidate K with BIC selection (warm-started and
  fresh fits for the previous K); label smoothing; rotated copies; Gaussian dropout; categorical hinge loss; DDBI to pick
  the iteration; then subtomogram averaging (RELION 3.0) of each cluster and mapping back into the tomograms.
- **What our toy keeps:** every step of the loop at miniature scale, including BIC model selection, Hungarian
  relabelling, smoothed labels with α = 0.2·0.9ᵗ, noise-filled rotations, hinge loss, DDBI and the >99.9 % stopping rule.
- **What it leaves out:** network size, 24³ inputs, M = 20 iterations, the candidate set {5, 10, 20} of the code (2–20 in
  the paper), the separate warm-up of a new output layer when K changes, RELION averaging, and all real data.
- **Result reported:** on simulated data DISCA reached 99.70 % accuracy versus 83.95 % for template matching (SNR 0.1,
  ±60°), with homogeneity above 0.8 in all settings (a convolutional-autoencoder baseline stayed below 0.55). On
  *M. pneumoniae* (EMPIAR-10499; 198,715 subtomograms from 65 tomograms) it gave a 14.17 Å ribosome average and a ribosome
  F1 of 0.893 versus 0.766 for raw template matching (false-positive rate 9.3 %, false-negative rate 12.1 %); five
  experimental datasets were analysed in total (*M. pneumoniae*, rat neurons, *Synechocystis*, mouse embryonic
  fibroblasts, monkey kidney cells). Sorting ~200,000 subtomograms took under a day.
- **Code:** [AITom `classify/deep/unsupervised/disca`](https://github.com/xulabs/aitom/tree/master/aitom/classify/deep/unsupervised/disca)
  (Keras and PyTorch versions).

### 📄 Paper connection — Jim-Net (Zeng, Howe & Xu, ICCV 2021)

**Zeng X, Howe G, Xu M.** *End-to-end robust joint unsupervised image alignment and clustering.* ICCV 2021, pp. 3854–3866.
[doi:10.1109/ICCV48922.2021.00383](https://doi.org/10.1109/ICCV48922.2021.00383).
- **What the paper does:** shared features feed a coarse-to-fine alignment module and a clustering branch; GMM
  pseudo-labels select same-cluster pairs for training (Algorithm 1); the 3D alignment loss is one minus a constrained
  correlation over Fourier coefficients observed in both volumes, low-pass filtered.
- **Results reported:** see Section 2; also 2D image matching on PF-PASCAL (PCK 74.8) and patterns in *Synechocystis*
  cells (phycobilisome arrays, 70S ribosomes, membranes; ribosome average at 36.48 Å).
- **Code:** [AITom `align/deep/jim`](https://github.com/xulabs/aitom/tree/master/aitom/align/deep/jim) (2D only).

### 📄 Paper connection — class discovery with semantic prototypes (Nair *et al.*, Brief. Bioinform. 2025)

**Nair A, Li X, Solanki B, Mukhopadhyay S, Jha A, Uddin MR, Singha M, Banerjee B, Xu M.** *Towards molecular structure
discovery from cryo-ET density volumes via modelling auxiliary semantic prototypes.* Briefings in Bioinformatics
26(1):bbae570 (published online November 2024). [doi:10.1093/bib/bbae570](https://doi.org/10.1093/bib/bbae570).
- **Setting:** generalized class discovery — semi-supervised (base classes labelled), not fully unsupervised.
- **What the paper does:** inflates a 2D self-supervised DINO ViT to 3D; refines CLIP text embeddings of class names with a
  graph convolutional network; represents samples by similarities to labelled-class embeddings; semi-supervised k-means
  with EMA-updated labelled prototypes and Hungarian matching; semantic-aware soft prototype assignment. ⚠️ The exact loss
  equations could not be extracted when we wrote this notebook — read the paper.
- **Our toy keeps:** semi-supervised k-means with pinned labelled points and All/Known/New evaluation. It leaves out all
  representation-learning and text/prototype components.

### 📄 Paper connection — unsupervised multi-task learning (Zhu *et al.*, ICIP 2022)

**Zhu H, Wang C, Wang Y, Fan Z, Uddin MR, Gao X, Zhang J, Zeng X, Xu M.** *Unsupervised multi-task learning for 3D
subtomogram image alignment, clustering and segmentation.* IEEE ICIP 2022, pp. 2751–2755.
[doi:10.1109/ICIP46576.2022.9897919](https://doi.org/10.1109/ICIP46576.2022.9897919).
- ⚠️ We could not access the text of this paper. From its title it trains one unsupervised network to align, cluster and
  segment subtomograms jointly — a natural extension of the Jim-Net idea. Read the paper before relying on any details.

## ✅ Checkpoint

1. Why does clustering raw voxels mix up different molecules?
   <details><summary>Answer</summary>Differences in pose (position, orientation), missing-wedge direction and noise are larger than the differences between molecules, so the raw-voxel distances are dominated by nuisance factors.</details>
2. How does YOPO obtain translation invariance?
   <details><summary>Answer</summary>Global max pooling after every convolution layer: the maximum response of a filter does not depend on where the pattern occurs.</details>
3. What does BIC trade off when choosing K?
   <details><summary>Answer</summary>Goodness of fit (log-likelihood) against model complexity (number of parameters × ln n); lower BIC is better.</details>
4. Why smooth the labels, and why let α shrink over iterations?
   <details><summary>Answer</summary>Early pseudo-labels are unreliable; smoothing prevents the network from fitting their errors too confidently. As clusters improve, sharper labels are justified.</details>
5. DDBI is low when … ? Why can DISCA use it but not ARI?
   <details><summary>Answer</summary>When clusters are compact relative to their separation. ARI needs ground-truth labels; DDBI does not.</details>
6. What is the difference between clustering and generalized class discovery?
   <details><summary>Answer</summary>In class discovery some classes have labels; the method must assign their members correctly and also form new clusters for the unlabelled classes.</details>

## 🧠 Questions & discussion

1. DISCA's features are learned from its own pseudo-labels. How could this self-reinforcement go wrong, and which design
   choices guard against it?
2. Why is over-partitioning (too many clusters) safer than under-partitioning for structural biology?
3. Two clusters give nearly identical averages. What would you do next, and what biological explanation might there be
   (think of conformations or bound partners)?
4. How would you check that a "new" cluster is a real complex and not junk (ice, gold beads, membrane fragments)?
5. Compare DISCA with Multi-Pattern Pursuit (Module 14): what is learned, what is searched, and what is scored?

## 🚀 Bonus — going further

- **Run the real thing.** AITom's `torch_disca.py` runs DISCA in PyTorch; try it on the simulated data from Module 08.
- **Scale up.** Use 24³ subtomograms at 1 nm and the full 10-layer YOPO on a Colab GPU. How do K and ARI change?
- **Align the clusters.** Replace the oracle in 4.2 with the constrained-correlation aligner of Module 11 or the
  maximum-likelihood averaging of Module 13.
- **Read** the DISCA supporting information for the exact DDBI definition and stopping rules.

## 📝 Lab note

```text
Module 20 — date:
Input: 560 simulated subtomograms (5 complexes, unequal abundances, SNR 0.1, ±60° wedge, 16³ at 2 nm)
Output: cluster labels per iteration, chosen K, DDBI, ARI/accuracy, cluster compositions, class-discovery accuracies
Method: DISCA-lite (YOPO → PCA → GMM + BIC → Hungarian → smoothed-label hinge training), semi-supervised k-means
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: Zeng et al. 2023 (DISCA); Zeng, Howe & Xu 2021 (Jim-Net); Nair et al. 2025
```

## 📚 References

1. Zeng X, Kahng A, Xue L, Mahamid J, Chang Y-W, Xu M. High-throughput cryo-ET structural pattern mining by unsupervised deep iterative subtomogram clustering. *PNAS* 120:e2213149120 (2023). doi:10.1073/pnas.2213149120
2. Zeng X, Howe G, Xu M. End-to-end robust joint unsupervised image alignment and clustering. *ICCV* 2021, pp. 3854–3866. doi:10.1109/ICCV48922.2021.00383
3. Nair A *et al.* Towards molecular structure discovery from cryo-ET density volumes via modelling auxiliary semantic prototypes. *Brief Bioinform* 26:bbae570 (2025). doi:10.1093/bib/bbae570
4. Zhu H *et al.* Unsupervised multi-task learning for 3D subtomogram image alignment, clustering and segmentation. *IEEE ICIP* 2022, pp. 2751–2755. doi:10.1109/ICIP46576.2022.9897919
5. Kim HH, Uddin MR, Xu M, Chang Y-W. Computational methods toward unbiased pattern mining and structure determination in cryo-electron tomography data. *J Mol Biol* 435:168068 (2023). doi:10.1016/j.jmb.2023.168068
6. Schwarz G. Estimating the dimension of a model. *Ann Stat* 6:461–464 (1978) (BIC).

## ➡️ Next

**[21 · Separating *what* a molecule is from *how* it is posed: Harmony and beyond](21_disentangling_identity_pose.ipynb)** —
instead of throwing pose away, learn it explicitly alongside identity. Course overview: [`README.md`](../README.md) ·
terms: [`GLOSSARY.md`](../GLOSSARY.md).